In [ ]:
import pandas as pd

wb_df = pd.read_csv("../data/clean_wb.csv")
imf_df = pd.read_csv("../data/clean_imf.csv")
oecd_df = pd.read_csv("../data/clean_oecd.csv")

: 

### Final Clean

Standardizing Columns

In [ ]:
def ensure_numeric_year(df, year_col="year"):
    df[year_col] = pd.to_numeric(df[year_col], errors="coerce")
    df = df[df[year_col].notna()].copy()
    df[year_col] = df[year_col].astype(int)
    return df


def ensure_numeric_value(df, value_col="value"):x
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    return df


def standardize_basic_schema(df, source_name):
    df = df.copy()

    # standardize year/value
    if "year" in df.columns:
        df = ensure_numeric_year(df, "year")
    if "value" in df.columns:
        df = ensure_numeric_value(df, "value")

    # clean text columns if they exist
    text_cols = ["country", "country_id", "indicator", "indicator_id"]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    # drop exact duplicates
    dedupe_cols = [
        c
        for c in ["country", "country_id", "year", "indicator", "indicator_id", "value"]
        if c in df.columns
    ]
    if dedupe_cols:
        df = df.drop_duplicates(subset=dedupe_cols)

    print(f"{source_name} standardized shape: {df.shape}")
    return df


wb_df = standardize_basic_schema(wb_df, "World Bank")
imf_df = standardize_basic_schema(imf_df, "IMF")
oecd_df = standardize_basic_schema(oecd_df, "OECD")

World Bank standardized shape: (13044108, 8)
IMF standardized shape: (8208, 116)
OECD standardized shape: (236918, 35)


In [56]:
print("WB columns:")
print(wb_df.columns.tolist())

print("\nIMF columns:")
print(imf_df.columns.tolist())

print("\nOECD columns:")
print(oecd_df.columns.tolist())

WB columns:
['country', 'country_id', 'indicator', 'indicator_id', 'year', 'value', 'decimal', 'source_file']

IMF columns:
['dataset', 'series_code', 'obs_measure', 'country', 'indicator', 'frequency', 'scale', 'decimals_displayed', 'functional_cat', 'int_acc_item', 'na_sto', 'gfs_sto', 'coicop_1999', 'trade_flow', 'commodity', 'soc_concepts', 'sector', 'accounting_entry', 'index_type', 'prices', 'statistical_measures', 'exrate', 'transformation', 'unit', 'reporting_period_type', 'overlap', 'country_update_date', 'doi', 'full_description', 'author', 'publisher', 'department', 'contact_point', 'topic', 'topic_dataset', 'keywords', 'keywords_dataset', 'language', 'publication_date', 'update_date', 'methodology', 'methodology_notes', 'access_sharing_level', 'access_sharing_notes', 'security_classification', 'short_source_citation', 'full_source_citation', 'license', 'suggested_citation', 'key_indicator', 'series_name', 'latest_actual_annual_data', 'historical_data_source', 'base_year', '

#### Choosing the indicators

In [57]:
# World Bank

wb_indicator_ids = {
    "gdp_growth": "NY.GDP.MKTP.KD.ZG",
    "inflation_cpi": "FP.CPI.TOTL.ZG",
    "unemployment": "SL.UEM.TOTL.ZS",
    "exports_pct_gdp": "NE.EXP.GNFS.ZS",
    "imports_pct_gdp": "NE.IMP.GNFS.ZS",
    "gov_consumption_pct_gdp": "NE.CON.GOVT.ZS",
    "current_account_pct_gdp": "BN.CAB.XOKA.GD.ZS",
}

# IMF

imf_indicator_keywords = {
    "real_gdp_growth_imf": [
        "real gdp growth",
        "gross domestic product, constant prices",
        "real gross domestic product",
    ],
    "inflation_imf": ["inflation", "consumer prices", "headline inflation", "cpi"],
    "unemployment_imf": ["unemployment rate", "unemployment"],
    "current_account_pct_gdp_imf": [
        "current account balance",
        "current account",
        "percent of gdp",
        "% of gdp",
    ],
    "gov_debt_pct_gdp_imf": [
        "general government gross debt",
        "gross debt",
        "government debt",
    ],
    "investment_imf": ["total investment", "gross capital formation", "investment"],
}

# OCED

oecd_indicator_keywords = {
    "cpi_oecd": ["consumer price", "cpi", "inflation"],
    "industrial_production_oecd": ["industrial production"],
    "unemployment_oecd": ["unemployment"],
    "interest_rate_oecd": ["interest rate", "short-term interest rate", "policy rate"],
    "consumer_confidence_oecd": ["consumer confidence"],
    "gdp_oecd": ["gross domestic product", "gdp"],
}

Preparing the datasets

In [58]:
# World Bank

wb_clean = wb_df.copy()

wb_clean["year"] = pd.to_numeric(wb_clean["year"], errors="coerce")
wb_clean["value"] = pd.to_numeric(wb_clean["value"], errors="coerce")
wb_clean["country"] = wb_clean["country"].astype(str).str.strip()
wb_clean["indicator"] = wb_clean["indicator"].astype(str).str.strip()
wb_clean["indicator_id"] = wb_clean["indicator_id"].astype(str).str.strip()

wb_clean = wb_clean.dropna(subset=["country", "year", "value", "indicator_id"])
wb_clean["year"] = wb_clean["year"].astype(int)

print("WB clean shape:", wb_clean.shape)

# IMF

imf_clean = imf_df.copy()

year_cols = [col for col in imf_clean.columns if str(col).isdigit()]
print("IMF year columns found:", year_cols[:5], "...", year_cols[-5:])

imf_clean = imf_clean.melt(
    id_vars=[col for col in imf_clean.columns if col not in year_cols],
    value_vars=year_cols,
    var_name="year",
    value_name="value",
)

imf_clean["year"] = pd.to_numeric(imf_clean["year"], errors="coerce")
imf_clean["value"] = pd.to_numeric(imf_clean["value"], errors="coerce")
imf_clean["country"] = imf_clean["country"].astype(str).str.strip()

imf_clean["indicator_text"] = (
    imf_clean["series_name"].fillna("").astype(str)
    + " | "
    + imf_clean["indicator"].fillna("").astype(str)
    + " | "
    + imf_clean["full_description"].fillna("").astype(str)
).str.strip()

imf_clean = imf_clean.dropna(subset=["country", "year", "value"])
imf_clean["year"] = imf_clean["year"].astype(int)

print("IMF clean shape:", imf_clean.shape)


# OECD

oecd_clean = oecd_df.copy()

oecd_clean = oecd_clean.rename(columns={"reference_area": "country"})

oecd_clean["value"] = oecd_clean["obs_value"]

oecd_clean["raw_time"] = oecd_clean["time_period"].astype(str)
oecd_clean["year"] = oecd_clean["raw_time"].str.extract(r"(\d{4})")

oecd_clean["year"] = pd.to_numeric(oecd_clean["year"], errors="coerce")
oecd_clean["value"] = pd.to_numeric(oecd_clean["value"], errors="coerce")
oecd_clean["country"] = oecd_clean["country"].astype(str).str.strip()

oecd_clean["indicator_text"] = (
    oecd_clean["structure_name"].fillna("").astype(str)
    + " | "
    + oecd_clean["activity"].fillna("").astype(str)
    + " | "
    + oecd_clean["economic_activity"].fillna("").astype(str)
    + " | "
    + oecd_clean["measure"].fillna("").astype(str)
    + " | "
    + oecd_clean["unit_measure"].fillna("").astype(str)
    + " | "
    + oecd_clean["transformation"].fillna("").astype(str)
).str.strip()

oecd_clean = oecd_clean.dropna(subset=["country", "year", "value"]).copy()
oecd_clean["year"] = oecd_clean["year"].astype(int)

print("OECD clean shape:", oecd_clean.shape)
display(oecd_clean[["country", "raw_time", "year", "value", "indicator_text"]].head())

WB clean shape: (4164449, 8)
IMF year columns found: ['1980', '1981', '1982', '1983', '1984'] ... ['2026', '2027', '2028', '2029', '2030']
IMF clean shape: (354240, 68)
OECD clean shape: (236917, 39)


,country,raw_time,year,value,indicator_text
0,Croatia,2017-06,2017,103.31430,Composite leading indicators | _Z | Not applic...
1,China (People’s Republic of),2023-02,2023,96.18640,Composite leading indicators | _Z | Not applic...
2,Australia,1969-04,1969,99.92799,Composite leading indicators | _Z | Not applic...
3,Australia,1969-03,1969,99.96107,Composite leading indicators | _Z | Not applic...
4,Australia,1969-02,1969,100.06830,Composite leading indicators | _Z | Not applic...


In [59]:
# Helper Functions for Variable Selection


def filter_wb_by_indicator_ids(df, indicator_id_map):
    reverse_map = {v: k for k, v in indicator_id_map.items()}
    out = df[df["indicator_id"].isin(reverse_map.keys())].copy()
    out["selected_feature"] = out["indicator_id"].map(reverse_map)
    return out[["country", "year", "selected_feature", "value"]]


def filter_by_keywords(df, keyword_map, text_col="indicator_text"):
    out_frames = []

    for feature_name, keywords in keyword_map.items():
        pattern = "|".join(keywords)
        subset = df[df[text_col].str.contains(pattern, case=False, na=False)].copy()
        subset["selected_feature"] = feature_name
        out_frames.append(subset[["country", "year", "selected_feature", "value"]])

    if len(out_frames) == 0:
        return pd.DataFrame(columns=["country", "year", "selected_feature", "value"])

    out = pd.concat(out_frames, ignore_index=True)
    return out

### Filtering the Datasets

In [ ]:
wb_selected = filter_wb_by_indicator_ids(wb_clean, wb_indicator_ids)

imf_selected = imf_clean[
    imf_clean["indicator"].str.contains(
        "consumer price|unemployment", case=False, na=False
    )
].copy()

oecd_selected = oecd_clean[
    oecd_clean["indicator_text"].str.contains(
        "composite leading indicators", case=False, na=False
    )
].copy()


def assign_oecd_feature(text):
    text = str(text)

    if " | LI | " in text and text.endswith("| IX | IX"):
        return "cli_level_oecd"
    elif " | LI | " in text and text.endswith("| IX | GY"):
        return "cli_growth_oecd"
    elif " | CCICP | " in text:
        return "consumer_confidence_oecd"
    elif " | BCICP | " in text:
        return "business_confidence_oecd"
    else:
        return "other_cli_oecd"


oecd_selected["selected_feature"] = oecd_selected["indicator_text"].apply(
    assign_oecd_feature
)

oecd_selected = oecd_selected[
    oecd_selected["selected_feature"].isin(
        [
            "cli_level_oecd",
            "cli_growth_oecd",
            "consumer_confidence_oecd",
            "business_confidence_oecd",
        ]
    )
][["country", "year", "selected_feature", "value"]].copy()


def assign_imf_feature(text):
    text = str(text).lower()

    if "consumer price" in text:
        return "imf_inflation_imf"
    elif "unemployment" in text:
        return "imf_unemployment_imf"
    else:
        return None


imf_selected["selected_feature"] = imf_selected["indicator"].apply(assign_imf_feature)
imf_selected = imf_selected.dropna(subset=["selected_feature"])

imf_final = imf_selected.groupby(
    ["country", "year", "selected_feature"], as_index=False
)["value"].mean()
print("WB selected:", wb_selected.shape)
print("IMF selected:", imf_selected.shape)
print("OECD selected:", oecd_selected.shape)

display(wb_selected.head())
display(imf_selected.head())
display(oecd_selected.head())

WB selected: (66061, 4)
IMF selected: (41790, 69)
OECD selected: (105896, 4)


,country,year,selected_feature,value
11488,Albania,2024,current_account_pct_gdp,-2.395546
11489,Algeria,2024,current_account_pct_gdp,-1.022499
11491,Andorra,2024,current_account_pct_gdp,16.062674
11492,Angola,2024,current_account_pct_gdp,6.247440
11493,Antigua and Barbuda,2024,current_account_pct_gdp,-8.215438


,dataset,series_code,obs_measure,country,indicator,frequency,scale,decimals_displayed,functional_cat,int_acc_item,...,labor_sector_employment_type,fiscal_sector_general_government_composition,fiscal_sector_valuation_of_debt,fiscal_sector_instruments_included_in_gross_and_net_debt,trade_sector_oil_coverage,primary_domestic_currency,year,value,indicator_text,selected_feature
41,IMF.RES:WEO(9.0.0),G110.PCPIPCH.A,OBS_VALUE,Advanced Economies,"All Items, Consumer price index (CPI), Period ...",Annual,Units,Three,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1980,13.627,"All Items, Consumer price index (CPI), Period ...",imf_inflation_imf
62,IMF.RES:WEO(9.0.0),G001.PCPIPCH.A,OBS_VALUE,World,"All Items, Consumer price index (CPI), Period ...",Annual,Units,Three,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1980,17.260,"All Items, Consumer price index (CPI), Period ...",imf_inflation_imf
133,IMF.RES:WEO(9.0.0),AUT.PCPIEPCH.A,OBS_VALUE,Austria,"All Items, Consumer price index (CPI), End-of-...",Annual,Units,Three,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Euro,1980,6.661,"All Items, Consumer price index (CPI), End-of-...",imf_inflation_imf
136,IMF.RES:WEO(9.0.0),AUT.PCPIPCH.A,OBS_VALUE,Austria,"All Items, Consumer price index (CPI), Period ...",Annual,Units,Three,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Euro,1980,6.325,"All Items, Consumer price index (CPI), Period ...",imf_inflation_imf
148,IMF.RES:WEO(9.0.0),GBR.PCPI.A,OBS_VALUE,United Kingdom,"All Items, Consumer price index (CPI), Period ...",Annual,Units,Three,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,British pound,1980,31.268,"All Items, Consumer price index (CPI), Period ...",imf_inflation_imf


,country,year,selected_feature,value
0,Croatia,2017,business_confidence_oecd,103.31430
1,China (People’s Republic of),2023,consumer_confidence_oecd,96.18640
2805,Australia,1971,cli_level_oecd,98.22379
2806,Australia,1971,cli_level_oecd,98.30476
2807,Australia,1971,cli_level_oecd,98.42854


In [96]:
# Collapse each source to unique country-year-feature rows


def collapse_to_unique_feature_rows(df, name="dataset"):
    dupes = df.duplicated(subset=["country", "year", "selected_feature"]).sum()
    print(f"{name} duplicate country-year-feature rows BEFORE collapse: {dupes}")

    df_collapsed = df.groupby(["country", "year", "selected_feature"], as_index=False)[
        "value"
    ].mean()

    dupes_after = df_collapsed.duplicated(
        subset=["country", "year", "selected_feature"]
    ).sum()
    print(f"{name} duplicate country-year-feature rows AFTER collapse: {dupes_after}")
    print(f"{name} collapsed shape: {df_collapsed.shape}\n")

    return df_collapsed


wb_selected = collapse_to_unique_feature_rows(wb_selected, "WB")
imf_selected = collapse_to_unique_feature_rows(imf_selected, "IMF")
oecd_selected = collapse_to_unique_feature_rows(oecd_selected, "OECD")

WB duplicate country-year-feature rows BEFORE collapse: 0
WB duplicate country-year-feature rows AFTER collapse: 0
WB collapsed shape: (66061, 4)

IMF duplicate country-year-feature rows BEFORE collapse: 26952
IMF duplicate country-year-feature rows AFTER collapse: 0
IMF collapsed shape: (14838, 4)

OECD duplicate country-year-feature rows BEFORE collapse: 99261
OECD duplicate country-year-feature rows AFTER collapse: 0
OECD collapsed shape: (6635, 4)



In [97]:
# Audit


def audit_selected(df, name):
    print(f"\n{name}")
    print("-" * 50)
    print("Shape:", df.shape)
    print("Countries:", df["country"].nunique())
    print("Years:", df["year"].nunique())
    print("Year range:", df["year"].min(), "to", df["year"].max())
    print("\nSelected feature counts:")
    print(df["selected_feature"].value_counts(dropna=False))
    print(
        "\nDuplicate country-year-feature rows:",
        df.duplicated(subset=["country", "year", "selected_feature"]).sum(),
    )


audit_selected(wb_selected, "World Bank")
audit_selected(imf_selected, "IMF")
audit_selected(oecd_selected, "OECD")


World Bank
--------------------------------------------------
Shape: (66061, 4)
Countries: 262
Years: 65
Year range: 1960 to 2024

Selected feature counts:
selected_feature
gdp_growth                 14133
inflation_cpi              11295
imports_pct_gdp            11167
exports_pct_gdp            11102
gov_consumption_pct_gdp    10756
current_account_pct_gdp     7608
Name: count, dtype: int64

Duplicate country-year-feature rows: 0

IMF
--------------------------------------------------
Shape: (14838, 4)
Countries: 210
Years: 51
Year range: 1980 to 2030

Selected feature counts:
selected_feature
imf_inflation_imf       9790
imf_unemployment_imf    5048
Name: count, dtype: int64

Duplicate country-year-feature rows: 0

OECD
--------------------------------------------------
Shape: (6635, 4)
Countries: 57
Years: 77
Year range: 1950 to 2026

Selected feature counts:
selected_feature
business_confidence_oecd    2307
consumer_confidence_oecd    2023
cli_level_oecd              1201
cli_gr

In [ ]:
# Pivot each source to wide format


def pivot_wide(df, prefix):
    wide = df.pivot(
        index=["country", "year"], columns="selected_feature", values="value"
    ).reset_index()

    wide.columns.name = None

    rename_cols = {
        col: f"{prefix}_{col}" for col in wide.columns if col not in ["country", "year"]
    }
    wide = wide.rename(columns=rename_cols)

    print(f"{prefix.upper()} wide shape: {wide.shape}")
    return wide


imf_wide = imf_final.pivot(
    index=["country", "year"], columns="selected_feature", values="value"
).reset_index()

imf_wide = imf_wide.rename(
    columns={"inflation": "imf_inflation_imf", "unemployment": "imf_unemployment_imf"}
)

wb_wide = pivot_wide(wb_selected, "wb")

oecd_wide = pivot_wide(oecd_selected, "oecd")

display(wb_wide.head())
display(imf_wide.head())
display(oecd_wide.head())

WB wide shape: (14590, 8)
OECD wide shape: (2725, 6)


,country,year,wb_current_account_pct_gdp,wb_exports_pct_gdp,wb_gdp_growth,wb_gov_consumption_pct_gdp,wb_imports_pct_gdp,wb_inflation_cpi
0,Afghanistan,2001,NaN,NaN,-9.431974,NaN,NaN,NaN
1,Afghanistan,2002,NaN,NaN,28.600001,NaN,NaN,NaN
2,Afghanistan,2003,NaN,NaN,8.832278,NaN,NaN,NaN
3,Afghanistan,2004,NaN,NaN,1.414118,NaN,NaN,NaN
4,Afghanistan,2005,NaN,NaN,11.229715,NaN,NaN,12.686269


selected_feature,country,year,imf_inflation_imf,imf_unemployment_imf
0,ASEAN-5,1980,16.6340,NaN
1,ASEAN-5,1981,11.2365,NaN
2,ASEAN-5,1982,7.4570,NaN
3,ASEAN-5,1983,8.1125,NaN
4,ASEAN-5,1984,13.6945,NaN


,country,year,oecd_business_confidence_oecd,oecd_cli_growth_oecd,oecd_cli_level_oecd,oecd_consumer_confidence_oecd
0,Australia,1966,100.060251,NaN,72.803594,NaN
1,Australia,1967,100.456148,6.408959,73.720694,NaN
2,Australia,1968,100.474110,5.515659,73.941512,NaN
3,Australia,1969,101.221983,6.816167,75.260537,NaN
4,Australia,1970,100.542225,4.060088,74.801131,NaN


In [99]:
# Quick Check again

print("WB sample countries:")
print(sorted(wb_wide["country"].dropna().unique())[:20])

print("\nIMF sample countries:")
print(sorted(imf_wide["country"].dropna().unique())[:20])

print("\nOECD sample countries:")
print(sorted(oecd_wide["country"].dropna().unique())[:20])

WB sample countries:
['Afghanistan', 'Africa Eastern and Southern', 'Africa Western and Central', 'Albania', 'Algeria', 'American Samoa', 'Andorra', 'Angola', 'Antigua and Barbuda', 'Arab World', 'Argentina', 'Armenia', 'Aruba', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas, The', 'Bahrain', 'Bangladesh', 'Barbados']

IMF sample countries:
['ASEAN-5', 'Advanced Economies', 'Afghanistan, Islamic Republic of', 'Albania', 'Algeria', 'Andorra, Principality of', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia, Republic of', 'Aruba, Kingdom of the Netherlands', 'Australia', 'Austria', 'Azerbaijan, Republic of', 'Bahamas, The', 'Bahrain, Kingdom of', 'Bangladesh', 'Barbados', 'Belarus, Republic of', 'Belgium']

OECD sample countries:
['Australia', 'Austria', 'Belgium', 'Brazil', 'Bulgaria', 'Canada', 'Chile', 'China (People’s Republic of)', 'Colombia', 'Costa Rica', 'Croatia', 'Czechia', 'Denmark', 'Estonia', 'Euro area (19 countries)', 'Euro area (20 countries)', 'European Union (27 

In [100]:
# Fixing country names to improve mergeability

country_map = {
    "China (People’s Republic of)": "China",
    "China (People's Republic of)": "China",
    "China, People's Republic of": "China",
    "Korea": "Korea, Rep.",
    "Korea, Republic of": "Korea, Rep.",
    "South Korea": "Korea, Rep.",
    "Iran, Islamic Rep.": "Iran",
    "Iran, Islamic Republic of": "Iran",
    "Russian Federation": "Russia",
    "Slovak Republic": "Slovakia",
    "Türkiye": "Turkey",
    "Egypt, Arab Rep.": "Egypt",
    "Egypt, Arab Republic of": "Egypt",
    "Venezuela, RB": "Venezuela",
    "Bahamas, The": "Bahamas",
    "Gambia, The": "Gambia",
    "Yemen, Rep.": "Yemen",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Lao PDR": "Laos",
    "Brunei Darussalam": "Brunei",
    "Czechia": "Czech Republic",
    "Hong Kong SAR, China": "Hong Kong SAR",
    "Hong Kong Special Administrative Region, People's Republic of China": "Hong Kong SAR",
    "Macao SAR, China": "Macao SAR",
    "Macao Special Administrative Region, People's Republic of China": "Macao SAR",
    "Afghanistan, Islamic Republic of": "Afghanistan",
}


def harmonize_country_names(df):
    df = df.copy()
    df["country"] = df["country"].replace(country_map)
    df["country"] = df["country"].astype(str).str.strip()
    return df


wb_wide = harmonize_country_names(wb_wide)
imf_wide = harmonize_country_names(imf_wide)
oecd_wide = harmonize_country_names(oecd_wide)

In [101]:
# Country overlap check

wb_countries = set(wb_wide["country"].dropna().unique())
imf_countries = set(imf_wide["country"].dropna().unique())
oecd_countries = set(oecd_wide["country"].dropna().unique())

print("WB countries:", len(wb_countries))
print("IMF countries:", len(imf_countries))
print("OECD countries:", len(oecd_countries))

print("\nWB ∩ IMF:", len(wb_countries.intersection(imf_countries)))
print("WB ∩ OECD:", len(wb_countries.intersection(oecd_countries)))
print("IMF ∩ OECD:", len(imf_countries.intersection(oecd_countries)))
print(
    "WB ∩ IMF ∩ OECD:",
    len(wb_countries.intersection(imf_countries).intersection(oecd_countries)),
)

WB countries: 262
IMF countries: 210
OECD countries: 57

WB ∩ IMF: 147
WB ∩ OECD: 45
IMF ∩ OECD: 39
WB ∩ IMF ∩ OECD: 38


In [102]:
# Merge all sources

merged_df = wb_wide.merge(imf_wide, on=["country", "year"], how="outer").merge(
    oecd_wide, on=["country", "year"], how="left"
)

print("Merged shape:", merged_df.shape)
display(merged_df.head())

Merged shape: (18166, 14)


,country,year,wb_current_account_pct_gdp,wb_exports_pct_gdp,wb_gdp_growth,wb_gov_consumption_pct_gdp,wb_imports_pct_gdp,wb_inflation_cpi,imf_inflation_imf,imf_unemployment_imf,oecd_business_confidence_oecd,oecd_cli_growth_oecd,oecd_cli_level_oecd,oecd_consumer_confidence_oecd
0,ASEAN-5,1980,NaN,NaN,NaN,NaN,NaN,NaN,16.6340,NaN,NaN,NaN,NaN,NaN
1,ASEAN-5,1981,NaN,NaN,NaN,NaN,NaN,NaN,11.2365,NaN,NaN,NaN,NaN,NaN
2,ASEAN-5,1982,NaN,NaN,NaN,NaN,NaN,NaN,7.4570,NaN,NaN,NaN,NaN,NaN
3,ASEAN-5,1983,NaN,NaN,NaN,NaN,NaN,NaN,8.1125,NaN,NaN,NaN,NaN,NaN
4,ASEAN-5,1984,NaN,NaN,NaN,NaN,NaN,NaN,13.6945,NaN,NaN,NaN,NaN,NaN


In [103]:
# Audit merged dataset

print("Merged shape:", merged_df.shape)
print("Countries:", merged_df["country"].nunique())
print("Years:", merged_df["year"].nunique())
print("Year range:", merged_df["year"].min(), "to", merged_df["year"].max())
print(
    "Duplicate country-year rows:",
    merged_df.duplicated(subset=["country", "year"]).sum(),
)

feature_cols = [c for c in merged_df.columns if c not in ["country", "year"]]

print("\nTotal feature columns:", len(feature_cols))
print("Sample feature columns:")
print(feature_cols[:20])

Merged shape: (18166, 14)
Countries: 325
Years: 71
Year range: 1960 to 2030
Duplicate country-year rows: 0

Total feature columns: 12
Sample feature columns:
['wb_current_account_pct_gdp', 'wb_exports_pct_gdp', 'wb_gdp_growth', 'wb_gov_consumption_pct_gdp', 'wb_imports_pct_gdp', 'wb_inflation_cpi', 'imf_inflation_imf', 'imf_unemployment_imf', 'oecd_business_confidence_oecd', 'oecd_cli_growth_oecd', 'oecd_cli_level_oecd', 'oecd_consumer_confidence_oecd']


In [104]:
# Filter out rows with very low feature coverage

feature_cols = [c for c in merged_df.columns if c not in ["country", "year"]]

merged_df["non_null_feature_count"] = merged_df[feature_cols].notna().sum(axis=1)
merged_df["feature_coverage_pct"] = merged_df["non_null_feature_count"] / len(
    feature_cols
)

print(merged_df["feature_coverage_pct"].describe())

# keep rows with at least 15% coverage
merged_df = merged_df[merged_df["feature_coverage_pct"] >= 0.15].copy()

print("Shape after row coverage filter:", merged_df.shape)

count    18166.000000
mean         0.394643
std          0.237917
min          0.083333
25%          0.166667
50%          0.416667
75%          0.583333
max          1.000000
Name: feature_coverage_pct, dtype: float64
Shape after row coverage filter: (15005, 16)


In [105]:
# Drop feature columns with too much missingness

missing_pct = merged_df.isna().mean().sort_values(ascending=False)
missing_summary = pd.DataFrame({"missing_pct": missing_pct})

display(missing_summary.head(25))

keep_always = ["country", "year", "non_null_feature_count", "feature_coverage_pct"]

cols_to_keep = [
    col
    for col in merged_df.columns
    if col in keep_always or merged_df[col].isna().mean() <= 0.50
]

merged_df = merged_df[cols_to_keep].copy()

print("Shape after dropping sparse columns:", merged_df.shape)

,missing_pct
oecd_cli_growth_oecd,0.946018
oecd_cli_level_oecd,0.940820
oecd_consumer_confidence_oecd,0.895701
oecd_business_confidence_oecd,0.875575
imf_unemployment_imf,0.663779
wb_current_account_pct_gdp,0.493036
imf_inflation_imf,0.474042
wb_gov_consumption_pct_gdp,0.283839
wb_exports_pct_gdp,0.260113
wb_inflation_cpi,0.258647


Shape after dropping sparse columns: (15005, 11)


In [106]:
# Impute missing values within each country

merged_df = merged_df.sort_values(["country", "year"]).copy()

meta_cols = ["country", "year", "non_null_feature_count", "feature_coverage_pct"]
feature_cols = [c for c in merged_df.columns if c not in meta_cols]

merged_df[feature_cols] = merged_df.groupby("country")[feature_cols].transform(
    lambda x: x.ffill().bfill()
)

# fallback: fill any remaining gaps with column median
for col in feature_cols:
    merged_df[col] = merged_df[col].fillna(merged_df[col].median())

print(
    "Remaining missing values in features:", merged_df[feature_cols].isna().sum().sum()
)

Remaining missing values in features: 0


In [107]:
### Drop aggregate / regional / analytical group entities

aggregate_entities = {
    "ASEAN-5",
    "Advanced Economies",
    "Africa Eastern and Southern",
    "Arab World",
    "Central Europe and the Baltics",
    "East Asia & Pacific",
    "East Asia & Pacific (IDA & IBRD countries)",
    "East Asia & Pacific (excluding high income)",
    "Emerging Market and Developing Economies",
    "Emerging and Developing Asia",
    "Emerging and Developing Europe",
    "Euro Area (EA)",
    "Euro area",
    "Europe & Central Asia",
    "Europe & Central Asia (IDA & IBRD countries)",
    "Europe & Central Asia (excluding high income)",
    "European Union",
    "European Union (EU)",
    "Fragile and conflict affected situations",
    "G7",
    "Latin America & Caribbean",
    "Latin America & Caribbean (excluding high income)",
    "Latin America & the Caribbean (IDA & IBRD countries)",
    "Latin America and the Caribbean (LAC)",
    "Lower middle income",
    "Middle East and Central Asia",
    "Middle East, North Africa, Afghanistan & Pakistan",
    "Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)",
    "Middle East, North Africa, Afghanistan & Pakistan (excluding high income)",
    "Other Advanced Economies (Advanced Economies excluding G7 and Euro Area countries)",
    "Other small states",
    "Small states",
    "South Asia",
    "South Asia (IDA & IBRD)",
    "Sub-Saharan Africa",
    "Sub-Saharan Africa (IDA & IBRD countries)",
    "Sub-Saharan Africa (SSA)",
    "Sub-Saharan Africa (excluding high income)",
    "Upper middle income",
    "World",
    "High income",
    "IBRD only",
    "IDA & IBRD total",
    "IDA blend",
    "IDA only",
    "IDA total",
    "Low & middle income",
    "Low income",
    "Micronesia, Federated States of",
    "Middle income",
}

before_rows = len(merged_df)
before_countries = merged_df["country"].nunique()

merged_df = merged_df[~merged_df["country"].isin(aggregate_entities)].copy()

print("Rows removed:", before_rows - len(merged_df))
print("Countries before:", before_countries)
print("Countries after:", merged_df["country"].nunique())

Rows removed: 2290
Countries before: 290
Countries after: 249


In [108]:
# Create lag features

base_feature_cols = [
    c
    for c in merged_df.columns
    if c not in ["country", "year", "non_null_feature_count", "feature_coverage_pct"]
]

for col in base_feature_cols:
    merged_df[f"{col}_lag1"] = merged_df.groupby("country")[col].shift(1)
    merged_df[f"{col}_lag2"] = merged_df.groupby("country")[col].shift(2)

lag_cols = [c for c in merged_df.columns if c.endswith("_lag1") or c.endswith("_lag2")]

merged_df[lag_cols] = merged_df.groupby("country")[lag_cols].transform(
    lambda x: x.ffill().bfill()
)

for col in lag_cols:
    merged_df[col] = merged_df[col].fillna(merged_df[col].median())

print("Lag features added.")
print("New shape:", merged_df.shape)

Lag features added.
New shape: (12715, 25)


In [109]:
# Create year-over-year change features

for col in base_feature_cols:
    merged_df[f"{col}_chg1"] = merged_df.groupby("country")[col].diff()

chg_cols = [c for c in merged_df.columns if c.endswith("_chg1")]

merged_df[chg_cols] = merged_df.groupby("country")[chg_cols].transform(
    lambda x: x.ffill().bfill()
)

for col in chg_cols:
    merged_df[col] = merged_df[col].fillna(merged_df[col].median())

print("Change features added.")
print("New shape:", merged_df.shape)

Change features added.
New shape: (12715, 32)


In [110]:
# Create recession target variable

recession_years = [2008, 2009, 2020]

merged_df["recession"] = merged_df["year"].isin(recession_years).astype(int)

print("Target distribution:")
print(merged_df["recession"].value_counts(dropna=False))
print(merged_df["recession"].value_counts(normalize=True))

Target distribution:
recession
0    11988
1      727
Name: count, dtype: int64
recession
0    0.942823
1    0.057177
Name: proportion, dtype: float64


In [111]:
# Final QA check

print("Final shape:", merged_df.shape)
print("Countries:", merged_df["country"].nunique())
print("Years:", merged_df["year"].nunique())
print(
    "Duplicate country-year rows:",
    merged_df.duplicated(subset=["country", "year"]).sum(),
)
print("Any missing values left:", merged_df.isna().sum().sum())

final_feature_cols = [
    c for c in merged_df.columns if c not in ["country", "year", "recession"]
]

print("\nNumber of model features:", len(final_feature_cols))
print("Sample final feature columns:")
print(final_feature_cols[:30])

Final shape: (12715, 33)
Countries: 249
Years: 71
Duplicate country-year rows: 0
Any missing values left: 0

Number of model features: 30
Sample final feature columns:
['wb_current_account_pct_gdp', 'wb_exports_pct_gdp', 'wb_gdp_growth', 'wb_gov_consumption_pct_gdp', 'wb_imports_pct_gdp', 'wb_inflation_cpi', 'imf_inflation_imf', 'non_null_feature_count', 'feature_coverage_pct', 'wb_current_account_pct_gdp_lag1', 'wb_current_account_pct_gdp_lag2', 'wb_exports_pct_gdp_lag1', 'wb_exports_pct_gdp_lag2', 'wb_gdp_growth_lag1', 'wb_gdp_growth_lag2', 'wb_gov_consumption_pct_gdp_lag1', 'wb_gov_consumption_pct_gdp_lag2', 'wb_imports_pct_gdp_lag1', 'wb_imports_pct_gdp_lag2', 'wb_inflation_cpi_lag1', 'wb_inflation_cpi_lag2', 'imf_inflation_imf_lag1', 'imf_inflation_imf_lag2', 'wb_current_account_pct_gdp_chg1', 'wb_exports_pct_gdp_chg1', 'wb_gdp_growth_chg1', 'wb_gov_consumption_pct_gdp_chg1', 'wb_imports_pct_gdp_chg1', 'wb_inflation_cpi_chg1', 'imf_inflation_imf_chg1']


In [112]:
# Save final modeling dataset

output_path = "../data/final_recession_panel.csv"
merged_df.to_csv(output_path, index=False)

print(f"Saved final dataset to: {output_path}")

Saved final dataset to: ../data/final_recession_panel.csv


In [113]:
import os

# Ensure the directory exists
os.makedirs("data", exist_ok=True)

# Save the file
merged_df.to_parquet("data/final_merged_panel.parquet", index=False)

In [ ]:
### Audit suspicious remaining country names

suspicious_keywords = [
    "income",
    "world",
    "europe",
    "asia",
    "africa",
    "caribbean",
    "union",
    "area",
    "economies",
    "states",
    "small",
    "ida",
    "ibrd",
]

suspicious_countries = sorted(
    [
        c
        for c in merged_df["country"].dropna().unique()
        if any(k in c.lower() for k in suspicious_keywords)
    ]
)

print("Suspicious remaining names:")
for c in suspicious_countries:
    print(c)

Suspicious remaining names:
Africa Western and Central
Caribbean small states
Central African Republic
Pacific island small states
South Africa
Trinidad and Tobago
United States


In [ ]:
### Check duplicate country-year rows after harmonization

dupes = merged_df[
    merged_df.duplicated(subset=["country", "year"], keep=False)
].sort_values(["country", "year"])

print("Duplicate country-year rows:", dupes.shape[0])
display(dupes.head(20))

Duplicate country-year rows: 0


,country,year,wb_current_account_pct_gdp,wb_exports_pct_gdp,wb_gdp_growth,wb_gov_consumption_pct_gdp,wb_imports_pct_gdp,wb_inflation_cpi,imf_inflation_imf,non_null_feature_count,...,imf_inflation_imf_lag1,imf_inflation_imf_lag2,wb_current_account_pct_gdp_chg1,wb_exports_pct_gdp_chg1,wb_gdp_growth_chg1,wb_gov_consumption_pct_gdp_chg1,wb_imports_pct_gdp_chg1,wb_inflation_cpi_chg1,imf_inflation_imf_chg1,recession


In [ ]:
[c for c in merged_df.columns if "infl" in c.lower() or "unemp" in c.lower()]

['wb_inflation_cpi',
 'imf_inflation_imf',
 'wb_inflation_cpi_lag1',
 'wb_inflation_cpi_lag2',
 'imf_inflation_imf_lag1',
 'imf_inflation_imf_lag2',
 'wb_inflation_cpi_chg1',
 'imf_inflation_imf_chg1']